# # Hazard-prioritisation module (Kaggle)
## 1. Setup 
- `MODE = "trial"` runs only the 20 visual-check clips; `"full"` runs the whole subset.
- `RESUME = True` loads saved results of sections 3–7 instead of running the models (cells marked `%%heavy` are skipped).
- `SHOW_VIZ = True` draws the visual checks (cells marked `%%viz`) on the 20 clips.

In [ ]:
!pip install -q ultralytics "transformers>=4.45" accelerate

In [ ]:
import json, random, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from IPython.core.magic import register_cell_magic
from IPython.display import FileLink, display

SUBSET = "check_01"
MODE = "full"
RESUME = False
SHOW_VIZ = False
SEED = 42

DATA_DIR = Path("/kaggle/input/datasets/amynguyen05/wad-check-01/wad_check_01")    
RESULTS_IN = Path("/kaggle/input/datasets/amynguyen05/wad-test-results") # saved results, used only when RESUME

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAMES_DIR = DATA_DIR / "frames"
OUT_DIR = Path(f"/kaggle/working/outputs/{SUBSET}")
SAVED = next(RESULTS_IN.rglob("hints.jsonl")).parent if RESUME else OUT_DIR
CACHE_DIR = next(RESULTS_IN.rglob("seg")).parent if RESUME else Path(f"/kaggle/working/cache/{SUBSET}")
VIZ_DIR = OUT_DIR / "viz"
for d in (OUT_DIR, CACHE_DIR, VIZ_DIR):
    d.mkdir(parents=True, exist_ok=True)

@register_cell_magic
# heavy: runs model/long loop (skipped when resume)
# viz: visual checks
def heavy(line, cell):
    if not RESUME:
        get_ipython().run_cell(cell)

@register_cell_magic
def viz(line, cell):
    if SHOW_VIZ:
        get_ipython().run_cell(cell)

### Resume
- Load the saved results of sections 3-7.

In [ ]:
if RESUME:
    det = pd.read_parquet(SAVED / "detections.parquet")
    surfaces = pd.read_parquet(SAVED / "surfaces.parquet")
    obj_depth = pd.read_parquet(SAVED / "object_depth.parquet")
    geo = pd.read_parquet(SAVED / "frame_geometry.parquet")
    GEO = geo.set_index(["clip_id", "frame"])
    objects = pd.read_parquet(SAVED / "objects.parquet")
    ego = pd.read_parquet(SAVED / "ego_motion.parquet")
    motion8 = pd.read_parquet(SAVED / "motion_frame8.parquet")

## 2. Load the subset
- The clip list, the 20 visual-check clips (the same clips after every stage) and the 3×3 frame grid used by every stage.
- The human reminder is shown above the grids for comparison only; it is never passed to any pipeline step.

### 2.1. Clips to run and clips to visualise

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv")
assert all((FRAMES_DIR / fp / f"{k}.jpg").exists() for fp in manifest.frame_path for k in range(9))

# 20 visual-check clips across the three response types
VIZ_PLAN = {"clear": 2, "hazard_redirect": 9, "hazard_continue": 9}
VIZ_CLIPS = pd.concat([manifest[manifest.response_type == rt].sample(n, random_state=SEED)
                       for rt, n in VIZ_PLAN.items()]).frame_path.tolist()
RUN_CLIPS = VIZ_CLIPS if MODE == "trial" else manifest.frame_path.tolist()

# human reminders
REMINDER = pd.read_json(DATA_DIR / "annotations.jsonl", lines=True).set_index("frame_path")["alter"].to_dict()
print(f"{SUBSET}: {len(manifest)} clips | running {len(RUN_CLIPS)} | visual checks on {len(VIZ_CLIPS)}")

### 2.2. Frame-grid helper

In [ ]:
def frame_path(clip, k):
    return FRAMES_DIR / clip / f"{k}.jpg"

def show_clip(clip, stage, draw=None, note=""): # 9 frames (3x3 grid)
    w, h = Image.open(frame_path(clip, 0)).size
    fig, axs = plt.subplots(3, 3, figsize=(15, 15 * h / w * 0.95 + 1))
    for k, ax in enumerate(axs.flat):
        ax.imshow(Image.open(frame_path(clip, k)))
        if draw is not None:
            draw(ax, clip, k)
        ax.set_title(f"frame {k}   t = {k / 3:.2f} s", fontsize=10)
        ax.axis("off")
    fig.suptitle(f"{clip}\nreference (display only): {REMINDER.get(clip, '')}" + (f"\n{note}" if note else ""),
                 fontsize=10, y=0.995)
    plt.tight_layout()
    out = VIZ_DIR / stage / f"{clip.replace('.frame', '')}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=80)
    plt.show(); plt.close(fig)

## 3. Object detection and tracking (YOLOE)
- Open-vocabulary detection on all 9 frames, with text prompts mapped to our hazard categories, then tracking across frames.
- Output: `detections.parquet`, one row per object per frame.

### 3.1. Text prompts and hazard categories

In [ ]:
PROMPTS = {
    # prompt word : category (same names as the reference labels)
    "person": "pedestrian", "child": "pedestrian",
    "car": "vehicle", "bus": "vehicle", "truck": "vehicle", "van": "vehicle", "tricycle": "vehicle",
    "bicycle": "two_wheeler", "motorcycle": "two_wheeler", "scooter": "two_wheeler",
    "pole": "pole", "pillar": "pole", "street lamp": "pole", "traffic light": "pole",
    "tree": "tree_plant", "potted plant": "tree_plant", "bush": "tree_plant",
    "sign": "sign", "signboard": "sign", "billboard": "sign",
    "fence": "fence_railing", "railing": "fence_railing",
    "traffic cone": "barrier_cone", "barrier": "barrier_cone", "bollard": "barrier_cone",
    "chair": "furniture_stall", "bench": "furniture_stall", "table": "furniture_stall",
    "market stall": "furniture_stall", "cart": "furniture_stall",
    "stairs": "stairs_step", "curb": "curb_edge", "puddle": "water",
    "pothole": "hole_uneven", "speed bump": "hole_uneven",
    "wall": "wall_door", "door": "wall_door",
    "dog": "animal",
    "trash can": "street_object", "box": "street_object", "vending machine": "street_object",
    "statue": "street_object", "suitcase": "street_object",
}
PROMPT_WORDS = list(PROMPTS)
print(f"{len(PROMPT_WORDS)} prompts -> {len(set(PROMPTS.values()))} categories")

### 3.2. Load YOLOE

In [ ]:
%%heavy
from ultralytics import YOLOE
DET_MODEL = "yoloe-11l-seg.pt" # masks and boxes
DET_IMGSZ = 1280 # higher resolution for small, distant
DET_CONF = 0.15 # keep low confidence
det_model = YOLOE(DET_MODEL)
det_model.set_classes(PROMPT_WORDS)

### 3.3. Detect objects in all 9 frames (resumes if interrupted)

In [ ]:
%%heavy
DET_FILE = OUT_DIR / "detections.parquet"
done = pd.read_parquet(DET_FILE) if DET_FILE.exists() else pd.DataFrame()
done_clips = set(done.clip_id) if len(done) else set()
todo = [c for c in RUN_CLIPS if c not in done_clips]
print(f"{len(done_clips)} clips done, {len(todo)} to run")

rows, t0 = [], time.time()
for i, clip in enumerate(todo, 1):
    results = det_model.predict([str(frame_path(clip, k)) for k in range(9)],
                                imgsz=DET_IMGSZ, conf=DET_CONF, verbose=False)
    for k, r in enumerate(results):
        H, W = r.orig_shape
        if r.boxes is None or len(r.boxes) == 0:
            continue
        polys = r.masks.xy if r.masks is not None else [None] * len(r.boxes)
        for box, conf, cls, poly in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                        r.boxes.cls.cpu().numpy().astype(int), polys):
            x1, y1, x2, y2 = box
            if poly is not None and len(poly) >= 3:
                area = cv2.contourArea(poly.astype(np.float32))
                y_bot = poly[:, 1].max()
                x_bot = poly[poly[:, 1] >= y_bot - 0.02 * H, 0].mean()
            else:
                area, y_bot, x_bot = (x2 - x1) * (y2 - y1), y2, (x1 + x2) / 2
            word = PROMPT_WORDS[cls]
            rows.append(dict(clip_id=clip, frame=k, prompt=word, category=PROMPTS[word], conf=float(conf),
                             x1=float(x1), y1=float(y1), x2=float(x2), y2=float(y2),
                             mask_area=float(area) / (W * H), ground_x=float(x_bot), ground_y=float(y_bot),
                             img_w=W, img_h=H))
    if i % 50 == 0 or i == len(todo):
        det = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        det.to_parquet(DET_FILE, index=False)
        print(f"  {i}/{len(todo)} clips | {(time.time() - t0) / i:.1f} s per clip")

det = pd.read_parquet(DET_FILE)
print(f"{len(det)} detections in {det.clip_id.nunique()} clips")

### 3.4. Link the same object across frames (tracking)

In [ ]:
%%heavy
from scipy.optimize import linear_sum_assignment
MAX_GAP = 2  # a track can be missing for up to 2 frames (0.67 s) and still continue
MAX_JUMP = 1.0 # max centre movement between matched frames, in units of the object's box size

def box_size(r): return max(r.x2 - r.x1, r.y2 - r.y1)
def centre(r): return np.array([(r.x1 + r.x2) / 2, (r.y1 + r.y2) / 2])

det = det.sort_values(["clip_id", "frame"]).reset_index(drop=True)
det["track_id"] = -1
for clip, g in det.groupby("clip_id"):
    tracks, next_id = [], 0 # dict(id, category, last_frame, last_row)
    for k in range(9):
        cur = g[g.frame == k]
        alive = [t for t in tracks if k - t["last_frame"] <= MAX_GAP + 1]
        cost = np.full((len(alive), len(cur)), 1e6)
        for a, t in enumerate(alive):
            for b, (_, r) in enumerate(cur.iterrows()):
                if r.category != t["category"]:
                    continue
                jump = np.linalg.norm(centre(r) - centre(t["last_row"])) / max(box_size(t["last_row"]), 1)
                if jump <= MAX_JUMP * (k - t["last_frame"]):
                    cost[a, b] = jump
        matched = set()
        if len(alive) and len(cur):
            for a, b in zip(*linear_sum_assignment(cost)):
                if cost[a, b] < 1e6:
                    idx = cur.index[b]
                    det.at[idx, "track_id"] = alive[a]["id"]
                    alive[a].update(last_frame=k, last_row=cur.loc[idx])
                    matched.add(idx)
        for idx, r in cur.iterrows():
            if idx not in matched:
                det.at[idx, "track_id"] = next_id
                tracks.append(dict(id=next_id, category=r.category, last_frame=k, last_row=r))
                next_id += 1

det["track_len"] = det.groupby(["clip_id", "track_id"]).frame.transform("nunique")
det.to_parquet(DET_FILE, index=False)
print(f"{det.groupby(['clip_id', 'track_id']).ngroups} tracks | median track length: {det.track_len.median():.0f} frames")

### 3.5. Visual check: detections

In [ ]:
%%viz
import matplotlib.patches as patches
CAT_COLORS = dict(zip(sorted(set(PROMPTS.values())), plt.cm.tab20.colors))

def draw_detections(ax, clip, k):
    for _, r in det[(det.clip_id == clip) & (det.frame == k)].iterrows():
        c = CAT_COLORS[r.category]
        ax.add_patch(patches.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, ec=c, lw=1.5))
        ax.plot(r.ground_x, r.ground_y, "o", color=c, ms=3)
        ax.text(r.x1, r.y1 - 2, f"{r.prompt} #{r.track_id} {r.conf:.2f}", color="white", fontsize=6,
                bbox=dict(facecolor=c, alpha=0.8, pad=0.5, lw=0))

for clip in VIZ_CLIPS:
    n8 = ((det.clip_id == clip) & (det.frame == 8)).sum()
    show_clip(clip, stage="02_detection", draw=draw_detections, note=f"{n8} objects detected in frame 8")

## 4. Surface segmentation (SegFormer-B5, ADE20K)
- Every pixel of all 9 frames is labelled and grouped into walking-relevant surfaces: walk, road, terrain, water, stairs, fence, wall.
- Outputs: label maps in `cache/seg/` and `surfaces.parquet`.

### 4.1. Load the model and define the surface groups

In [ ]:
import re
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

SEG_MODEL = "nvidia/segformer-b5-finetuned-ade-640-640"
seg_proc = SegformerImageProcessor.from_pretrained(SEG_MODEL)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL).to(DEVICE).eval()

SURFACE_GROUPS = { # group id: (name, ADE20K class names, colour)
    1: ("walk",    ["floor", "sidewalk", "path", "rug", "runway"],                       (0.10, 0.75, 0.25)),
    2: ("road",    ["road"],                                                             (0.95, 0.80, 0.10)),
    3: ("terrain", ["grass", "earth", "field", "sand", "dirt track", "land"],             (0.60, 0.90, 0.40)),
    4: ("water",   ["water", "sea", "river", "lake", "swimming pool", "fountain", "waterfall"], (0.10, 0.40, 0.95)),
    5: ("stairs",  ["stairs", "stairway", "step", "escalator"],                          (1.00, 0.50, 0.00)),
    6: ("fence",   ["fence", "railing", "bannister"],                                    (0.60, 0.20, 0.80)),
    7: ("wall",    ["wall", "building", "house", "skyscraper", "door", "column"],        (0.50, 0.50, 0.50)),
}
LUT = np.zeros(len(seg_model.config.id2label), dtype=np.uint8)          # ADE20K class id -> group id (0 = other)
for cid, name in seg_model.config.id2label.items():
    names = [n.strip() for n in re.split(r"[;,]", name)]
    for gid, (_, ade_names, _) in SURFACE_GROUPS.items():
        if any(n in ade_names for n in names):
            LUT[int(cid)] = gid

### 4.2. Segment all 9 frames

In [ ]:
%%heavy
SEG_W = 320 # saved label maps (enough for geometry, small on disk)
(CACHE_DIR / "seg").mkdir(exist_ok=True)

todo = [c for c in RUN_CLIPS if not (CACHE_DIR / "seg" / f"{c}.npz").exists()]
print(f"{len(RUN_CLIPS) - len(todo)} clips done, {len(todo)} to run")
t0 = time.time()
for i, clip in enumerate(todo, 1):
    imgs = [Image.open(frame_path(clip, k)).convert("RGB") for k in range(9)]
    W, H = imgs[0].size
    seg_h = round(SEG_W * H / W)
    inputs = seg_proc(images=imgs, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = seg_model(**inputs).logits  # (9, 150, 160, 160)
    up = F.interpolate(logits, size=(seg_h, SEG_W), mode="bilinear", align_corners=False)
    groups = LUT[up.argmax(1).cpu().numpy()]  # (9, seg_h, 320) group ids
    np.savez_compressed(CACHE_DIR / "seg" / f"{clip}.npz", groups=groups)
    if i % 50 == 0 or i == len(todo):
        print(f"  {i}/{len(todo)} clips | {(time.time() - t0) / i:.1f} s per clip")

### 4.3. Surface shares per frame

In [ ]:
%%heavy
rows = []
for clip in RUN_CLIPS:
    groups = np.load(CACHE_DIR / "seg" / f"{clip}.npz")["groups"]
    _, h, w = groups.shape
    for k in range(9):
        bottom = groups[k, h // 2:] # lower half of the image = the ground ahead
        centre = groups[k, h // 2:, w // 3: 2 * w // 3] # lower-middle third = roughly the walking path
        row = dict(clip_id=clip, frame=k)
        for gid, (g, _, _) in SURFACE_GROUPS.items():
            row[f"{g}_bottom"] = float((bottom == gid).mean())
            row[f"{g}_centre"] = float((centre == gid).mean())
        rows.append(row)
surf = pd.DataFrame(rows)
surf.to_parquet(OUT_DIR / "surfaces.parquet", index=False)

s8 = surf[surf.frame == 8]
print(f"frame 8: water in the lower middle in {(s8.water_centre > 0.02).sum()} clips, stairs in {(s8.stairs_centre > 0.02).sum()}")

### 4.4. Visual check: surfaces

In [ ]:
%%viz
from matplotlib.colors import ListedColormap
SEG_CMAP = ListedColormap([(0, 0, 0, 0)] + [(*c, 1) for _, _, c in SURFACE_GROUPS.values()])
LEGEND = "  ".join(f"{g}" + "■" for g, _, _ in SURFACE_GROUPS.values())

def draw_surfaces(ax, clip, k):
    groups = np.load(CACHE_DIR / "seg" / f"{clip}.npz")["groups"][k]
    W, H = Image.open(frame_path(clip, k)).size
    masked = np.ma.masked_equal(groups, 0)                            # leave "other" pixels uncoloured
    ax.imshow(masked, cmap=SEG_CMAP, vmin=0, vmax=len(SURFACE_GROUPS), alpha=0.45,
              extent=(0, W, H, 0), interpolation="nearest")

legend = "green=walk  yellow=road  light-green=terrain  blue=water  orange=stairs  purple=fence  grey=wall"
for clip in VIZ_CLIPS:
    show_clip(clip, stage="03_surfaces", draw=draw_surfaces, note=legend)

## 5. Depth (Depth Anything V2, metric outdoor)
- Depth for all 9 frames on the same 320-px grid as the segmentation, then a distance for each detected object.
- Outputs: depth maps in `cache/depth/` and `object_depth.parquet`. The depth scale is corrected in section 6.

### 5.1. Load the depth model

In [ ]:
%%heavy
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Base-hf"
depth_proc = AutoImageProcessor.from_pretrained(DEPTH_MODEL)
depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL).to(DEVICE).eval()

### 5.2. Depth for all 9 frames 

In [ ]:
%%heavy
(CACHE_DIR / "depth").mkdir(exist_ok=True)
todo = [c for c in RUN_CLIPS if not (CACHE_DIR / "depth" / f"{c}.npz").exists()]
print(f"{len(RUN_CLIPS) - len(todo)} clips done, {len(todo)} to run")
t0 = time.time()
for i, clip in enumerate(todo, 1):
    imgs = [Image.open(frame_path(clip, k)).convert("RGB") for k in range(9)]
    W, H = imgs[0].size
    seg_h = round(SEG_W * H / W) # same grid as the segmentation maps
    inputs = depth_proc(images=imgs, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        pred = depth_model(**inputs).predicted_depth # (9, h, w), metres
    depth = F.interpolate(pred.unsqueeze(1), size=(seg_h, SEG_W), mode="bilinear", align_corners=False)[:, 0]
    np.savez_compressed(CACHE_DIR / "depth" / f"{clip}.npz", depth=depth.cpu().numpy().astype(np.float16))
    if i % 50 == 0 or i == len(todo):
        print(f"  {i}/{len(todo)} clips | {(time.time() - t0) / i:.1f} s per clip")

### 5.3. Depth of each detected object

In [ ]:
%%heavy
def depth_at(dmap, x, y, img_w, img_h, r=2):
    """Median depth in a small window around image point (x, y); dmap is the 320-wide grid."""
    h, w = dmap.shape
    u, v = int(np.clip(x / img_w * w, 0, w - 1)), int(np.clip(y / img_h * h, 0, h - 1))
    return float(np.median(dmap[max(v - r, 0): v + r + 1, max(u - r, 0): u + r + 1]))

rows = []
for clip, g in det[det.clip_id.isin(RUN_CLIPS)].groupby("clip_id"):
    depth = np.load(CACHE_DIR / "depth" / f"{clip}.npz")["depth"].astype(np.float32)
    for _, r in g.iterrows():
        d = depth[r.frame]
        h, w = d.shape
        # inner part of the box (shrunk by 25% each side) -> median depth of the object itself
        bx1, bx2 = r.x1 + 0.25 * (r.x2 - r.x1), r.x2 - 0.25 * (r.x2 - r.x1)
        by1, by2 = r.y1 + 0.25 * (r.y2 - r.y1), r.y2 - 0.25 * (r.y2 - r.y1)
        u1, u2 = int(bx1 / r.img_w * w), max(int(bx2 / r.img_w * w), int(bx1 / r.img_w * w) + 1)
        v1, v2 = int(by1 / r.img_h * h), max(int(by2 / r.img_h * h), int(by1 / r.img_h * h) + 1)
        rows.append(dict(clip_id=clip, frame=r.frame, track_id=r.track_id,
                         depth_ground=depth_at(d, r.ground_x, r.ground_y - 0.01 * r.img_h, r.img_w, r.img_h),
                         depth_object=float(np.median(d[v1:v2, u1:u2]))))
obj_depth = pd.DataFrame(rows)
obj_depth.to_parquet(OUT_DIR / "object_depth.parquet", index=False)
print(f"{len(obj_depth)} object depths | median at frame 8: {obj_depth[obj_depth.frame == 8].depth_ground.median():.1f} m (before scale correction)")

### 5.4. Visual check: depth

In [ ]:
%%viz
def draw_depth(ax, clip, k):
    depth = np.load(CACHE_DIR / "depth" / f"{clip}.npz")["depth"][k].astype(np.float32)
    W, H = Image.open(frame_path(clip, k)).size
    ax.imshow(depth, cmap="Spectral", vmin=0, vmax=20, alpha=0.55, extent=(0, W, H, 0))
    sub = obj_depth[(obj_depth.clip_id == clip) & (obj_depth.frame == k)].merge(
        det[["clip_id", "frame", "track_id", "prompt", "ground_x", "ground_y", "conf"]], on=["clip_id", "frame", "track_id"])
    for _, r in sub[sub.conf >= 0.3].iterrows():
        ax.text(r.ground_x, r.ground_y, f"{r.prompt} {r.depth_ground:.1f}m", color="white", fontsize=6,
                ha="center", va="bottom", bbox=dict(facecolor="black", alpha=0.6, pad=0.5, lw=0))

for clip in VIZ_CLIPS:
    show_clip(clip, stage="04_depth", draw=draw_depth,
              note="depth: red = near, blue = far (0-20 m); labels = distance at the object's ground point (conf >= 0.3)")

## 6. Scene geometry (all 9 frames)
- Ground plane per frame (RANSAC on walkable pixels); the depth scale is fixed by an assumed 1.5 m camera height.
- For every object: ground position, distance (m and steps), clock direction, height, and whether it is in the walking corridor (±0.6 m, up to 8 m) or flat (a shadow or marking).
- Outputs: `frame_geometry.parquet`, `objects.parquet`.

### 6.1. Camera assumptions and ground-plane tools

In [ ]:
HFOV_LANDSCAPE, HFOV_PORTRAIT = 70.0, 43.0 # assumed horizontal field of view (degrees); portrait = landscape's vertical FOV
CAM_HEIGHT = 1.5 # assumed camera height above the ground (m): chest/head-mounted
STEP_M = 0.7 # one walking step (m)
CORRIDOR_HALF = 0.6 # half-width of the walking corridor (m): body ~0.45 m + margin
CORRIDOR_LEN = 8.0 # how far ahead the corridor reaches (m)
WALKABLE = [1, 2, 3] # surface groups: walk, road, terrain
GROUND_LEVEL = {"curb_edge", "hole_uneven", "stairs_step", "water"} # flat hazards

def intrinsics(w, h): # focal length and image centre
    hfov = HFOV_LANDSCAPE if w >= h else HFOV_PORTRAIT
    return (w / 2) / np.tan(np.radians(hfov) / 2), w / 2, h / 2

def fit_ground_plane(depth, groups, f, cx, cy, rng, n_iter=200):
    # RANSAC plane through walkable pixels in the lower half.
    # Returns unit normal a (pointing up) and offset b (camera height in raw depth units) for the plane a.P + b = 0, or None if no good ground is visible
    h, w = depth.shape
    ys, xs = np.where(np.isin(groups, WALKABLE) & (np.arange(h)[:, None] >= h // 2))
    if len(ys) < 300:
        return None
    pick = rng.choice(len(ys), min(3000, len(ys)), replace=False)
    ys, xs = ys[pick], xs[pick]
    Z = depth[ys, xs]
    P = np.stack([(xs - cx) / f * Z, (ys - cy) / f * Z, Z], axis=1)
    thr = 0.03 * np.median(Z)
    best = None
    for _ in range(n_iter):
        p = P[rng.choice(len(P), 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0])
        if np.linalg.norm(n) < 1e-9:
            continue
        n /= np.linalg.norm(n)
        inl = np.abs(P @ n - n @ p[0]) < thr
        if best is None or inl.sum() > best.sum():
            best = inl
    if best is None or best.sum() < 100:
        return None
    Q = P[best]
    c = Q.mean(0)
    a = np.linalg.svd(Q - c)[2][-1] # least-squares normal through the inliers
    b = -a @ c
    if b < 0:
        a, b = -a, -b                                # orient so the camera is on the positive (upper) side
    tilt = np.degrees(np.arccos(np.clip(-a[1], -1, 1)))   # angle between the plane normal and camera "up"
    if tilt > 40:
        return None                                  # not a ground-like plane (e.g. a wall)
    return a, b

def ground_axes(a):
    """Forward and right directions along the ground plane, in camera coordinates."""
    fwd = np.array([0, 0, 1.0]) - a[2] * a
    fwd /= np.linalg.norm(fwd)
    return fwd, np.cross(fwd, a)

### 6.2. Ground plane and depth scale for every frame

In [ ]:
%%heavy
geo_rows = []
rng_geo = np.random.default_rng(SEED)
for clip in RUN_CLIPS:
    depth = np.load(CACHE_DIR / "depth" / f"{clip}.npz")["depth"].astype(np.float32)
    groups = np.load(CACHE_DIR / "seg" / f"{clip}.npz")["groups"]
    h, w = depth.shape[1:]
    f, cx, cy = intrinsics(w, h) # on the 320-px grid
    for k in range(9):
        fit = fit_ground_plane(depth[k], groups[k], f, cx, cy, rng_geo)
        geo_rows.append(dict(clip_id=clip, frame=k, plane_ok=fit is not None,
                             ax=fit[0][0] if fit else np.nan, ay=fit[0][1] if fit else np.nan,
                             az=fit[0][2] if fit else np.nan, cam_h_raw=fit[1] if fit else np.nan))
geo = pd.DataFrame(geo_rows)

# Depth scale per clip: assumed camera height / camera height measured in raw depth units (median over good frames)
geo["scale"] = CAM_HEIGHT / geo.groupby("clip_id").cam_h_raw.transform("median")
# Frames without a good plane borrow the clip's median plane orientation. Clips with none use a 10 deg downward tilt
for col in ["ax", "ay", "az"]:
    geo[col] = geo[col].fillna(geo.groupby("clip_id")[col].transform("median"))
default_a = np.array([0, -np.cos(np.radians(10)), -np.sin(np.radians(10))])
for i, col in enumerate(["ax", "ay", "az"]):
    geo[col] = geo[col].fillna(default_a[i])
geo["scale"] = geo.scale.fillna(geo.drop_duplicates("clip_id").scale.median()) # clips with no ground: typical scale
geo["pitch_deg"] = np.degrees(np.arcsin(np.clip(-geo.az, -1, 1))) # camera looking down = positive
geo.to_parquet(OUT_DIR / "frame_geometry.parquet", index=False)

print(f"good ground plane in {geo.plane_ok.mean():.0%} of frames | median depth scale: {geo.drop_duplicates('clip_id').scale.median():.2f}")

### 6.3. Position, distance, clock direction and corridor for every object

In [ ]:
%%heavy
GEO = geo.set_index(["clip_id", "frame"])
obj = det.merge(obj_depth, on=["clip_id", "frame", "track_id"])
out = []
for (clip, k), g in obj.groupby(["clip_id", "frame"]):
    gr = GEO.loc[(clip, k)]
    a = np.array([gr.ax, gr.ay, gr.az]); s = gr.scale
    fwd, right = ground_axes(a)
    foot = -CAM_HEIGHT * a # point on the ground right below the camera
    W, H = int(g.img_w.iloc[0]), int(g.img_h.iloc[0])
    f, cx, cy = intrinsics(W, H) # in full-image pixels
    for _, r in g.iterrows():
        ray = np.array([(r.ground_x - cx) / f, (r.ground_y - cy) / f, 1.0])
        z_depth = s * r.depth_ground
        z_inside = s * r.depth_object # depth of the object's own pixels
        cut_bottom = r.y2 >= 0.98 * H # feet below the frame
        denom = a @ ray
        P, source = None, "depth"
        if denom < -1e-6:
            P_plane = (-CAM_HEIGHT / denom) * ray # where the lowest visible point meets the ground
            if cut_bottom:
                z = min(z_inside, P_plane[2]) # base hidden: plane is only an upper bound
                P, source = z * ray, "depth (cut off)"
            elif 0.5 < P_plane[2] < 60 and 0.5 < P_plane[2] / max(z_depth, 1e-3) < 2.0:
                P, source = P_plane, "plane"
        if P is None:
            P = (z_inside if cut_bottom else z_depth) * ray
        lateral, forward = (P - foot) @ right, (P - foot) @ fwd
        dist = float(np.hypot(lateral, forward))
        angle = np.degrees(np.arctan2(lateral, forward))
        clock = int(round(angle / 30)) % 12 or 12
        top = s * r.depth_object * np.array([((r.x1 + r.x2) / 2 - cx) / f, (r.y1 - cy) / f, 1.0])
        height = float(a @ top + CAM_HEIGHT)
        width = float((r.x2 - r.x1) / f * max(P[2], 0.1))
        out.append(dict(clip_id=clip, frame=k, track_id=r.track_id, category=r.category, prompt=r.prompt,
                        conf=r.conf, dist_m=dist, lateral_m=float(lateral), forward_m=float(forward),
                        clock=clock, steps=dist / STEP_M, height_m=height, width_m=width, dist_source=source,
                        in_corridor=bool(0 < forward <= CORRIDOR_LEN and
                                         abs(lateral) - width / 2 <= CORRIDOR_HALF),
                        flat=bool(height < 0.2 and r.category not in GROUND_LEVEL and gr.plane_ok),
                        ground_x=r.ground_x, ground_y=r.ground_y, x1=r.x1, y1=r.y1, x2=r.x2, y2=r.y2))
objects = pd.DataFrame(out)
objects["step_bin"] = pd.cut(objects.steps, [-np.inf, 7.5, 12.5, 17.5, np.inf], labels=[5, 10, 15, 20]).astype(int)
objects.to_parquet(OUT_DIR / "objects.parquet", index=False)

o8 = objects[objects.frame == 8]
print(f"frame 8: {len(o8)} objects | in corridor {o8.in_corridor.sum()} | flat (ignored) {o8.flat.sum()} | distance from the ground plane {(o8.dist_source == 'plane').mean():.0%}")

### 6.4. Drawing helpers: corridor and labels on a frame

In [ ]:
def project(P, f, cx, cy):
    return f * P[0] / P[2] + cx, f * P[1] / P[2] + cy

def draw_geometry(ax, clip, k):
    gr = GEO.loc[(clip, k)]
    a = np.array([gr.ax, gr.ay, gr.az]); fwd, right = ground_axes(a); foot = -CAM_HEIGHT * a
    W, H = Image.open(frame_path(clip, k)).size
    f, cx, cy = intrinsics(W, H)
    for side in (-CORRIDOR_HALF, CORRIDOR_HALF): # corridor edges, 1 m to 8 m ahead
        pts = [project(foot + side * right + z * fwd, f, cx, cy) for z in np.linspace(1, CORRIDOR_LEN, 15)]
        ax.plot(*zip(*pts), color="cyan", lw=1.2, ls="--")
    for z in (2, 4, 6, 8): # distance ticks across the corridor
        p1, p2 = project(foot - CORRIDOR_HALF * right + z * fwd, f, cx, cy), project(foot + CORRIDOR_HALF * right + z * fwd, f, cx, cy)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="cyan", lw=0.6)
        ax.text(p2[0] + 4, p2[1], f"{z} m", color="cyan", fontsize=5, clip_on=True)
    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    for _, r in objects[(objects.clip_id == clip) & (objects.frame == k) & (objects.conf >= 0.3)].iterrows():
        if r.flat:
            ax.plot(r.ground_x, r.ground_y, "x", color="grey", ms=6, mew=1.5)
            continue
        c = "red" if r.in_corridor else "white"
        ax.text(r.ground_x, r.ground_y, f"{r.prompt} {r.clock}h {r.steps:.0f}st", color=c, fontsize=6, ha="center",
                va="bottom", bbox=dict(facecolor="black", alpha=0.65, pad=0.5, lw=0))

### 6.5. Visual check: corridor and labels on the frames

In [ ]:
%%viz
for clip in VIZ_CLIPS:
    show_clip(clip, stage="05_geometry", draw=draw_geometry,
              note="cyan = walking corridor (±0.6 m, 1-8 m); label = object, clock direction, steps; red = in corridor; grey x = flat (shadow/marking)")

### 6.6. Visual check: frame 8 with a bird's-eye view

In [ ]:
%%viz
SHOW_MAX = 12 # objects shown per clip (frame 8, nearest, anything in the corridor)

def scene_objects(clip):
    """Objects at frame 8 to show, numbered 1, 2, 3... from nearest to farthest."""
    o = objects[(objects.clip_id == clip) & ~objects.flat & (objects.conf >= 0.3)]
    o8 = o[o.frame == 8]
    o8 = pd.concat([o8[o8.in_corridor], o8[~o8.in_corridor].nsmallest(SHOW_MAX, "dist_m")]).drop_duplicates("track_id")
    o8 = o8.sort_values("dist_m").reset_index(drop=True)
    o8["num"] = np.arange(1, len(o8) + 1)
    return o, o8

def badge(ax, x, y, num, color, edge):
    ax.scatter(x, y, s=150, color=color, edgecolor=edge, linewidth=1.8, zorder=4, clip_on=True)
    ax.text(x, y, str(num), fontsize=7, ha="center", va="center", color="white", fontweight="bold", zorder=5,
            clip_on=True)

def show_scene(clip):
    o, o8 = scene_objects(clip)
    W, H = Image.open(frame_path(clip, 8)).size
    img_h = 12 * H / W # height of the camera row
    fig = plt.figure(figsize=(12, img_h + 7.5), layout="constrained")
    gs = fig.add_gridspec(2, 2, height_ratios=[img_h, 7.5], width_ratios=[1.3, 1])
    ax_img = fig.add_subplot(gs[0, :])  # row 1: camera view, full width
    ax_map = fig.add_subplot(gs[1, 0])  # row 2 left: bird's-eye map
    ax_key = fig.add_subplot(gs[1, 1]); ax_key.axis("off") # row 2 right: key and legend

    # Row 1: frame 8 with the corridor and numbered objects
    ax_img.imshow(Image.open(frame_path(clip, 8)))
    gr = GEO.loc[(clip, 8)]
    a = np.array([gr.ax, gr.ay, gr.az]); fwd, right = ground_axes(a); foot = -CAM_HEIGHT * a
    f, cx, cy = intrinsics(W, H)
    for side in (-CORRIDOR_HALF, CORRIDOR_HALF):
        pts = [project(foot + side * right + z * fwd, f, cx, cy) for z in np.linspace(1, CORRIDOR_LEN, 15)]
        ax_img.plot(*zip(*pts), color="cyan", lw=1.5, ls="--")
    for _, r in o8.iterrows():
        badge(ax_img, r.ground_x, r.ground_y, r.num, CAT_COLORS[r.category], "red" if r.in_corridor else "white")
    ax_img.set_xlim(0, W); ax_img.set_ylim(H, 0); ax_img.axis("off")
    ax_img.set_title("frame 8 (t = 2.67 s) — numbers = nearest first", fontsize=10)

    # Row 2: bird's-eye map, same numbers, path over the whole clip (frames 0-8)
    ax = ax_map
    ax.add_patch(patches.Rectangle((-CORRIDOR_HALF, 0), 2 * CORRIDOR_HALF, CORRIDOR_LEN, color="cyan", alpha=0.25))
    for deg in (-75, -45, -15, 15, 45, 75):
        ax.plot([0, 16 * np.sin(np.radians(deg))], [0, 16 * np.cos(np.radians(deg))], color="lightgrey", lw=0.6)
    for h_, deg in ((10, -60), (11, -30), (12, 0), (1, 30), (2, 60)):
        ax.text(13.5 * np.sin(np.radians(deg)), 13.5 * np.cos(np.radians(deg)), f"{h_} o'clock", fontsize=7,
                color="grey", ha="center", clip_on=True)
    for d in (2, 4, 6, 8, 10):                                  # distance rings
        ax.add_patch(patches.Arc((0, 0), 2 * d, 2 * d, theta1=10, theta2=170, color="lightgrey", lw=0.5))
        ax.text(d * 0.71 + 0.2, d * 0.71, f"{d} m", fontsize=6, color="grey")
    for _, r in o8.iterrows():
        c = CAT_COLORS[r.category]
        path = (o[o.track_id == r.track_id].sort_values("frame")[["lateral_m", "forward_m"]]
                .rolling(3, center=True, min_periods=1).median())
        if len(path) >= 3:
            ax.plot(path.lateral_m, path.forward_m, color=c, lw=1.5, alpha=0.6)
            ax.scatter(path.lateral_m.iloc[0], path.forward_m.iloc[0], s=12, color=c, alpha=0.6, marker="s")  # start
        badge(ax, r.lateral_m, r.forward_m, r.num, c, "red" if r.in_corridor else "white")
    ax.plot(0, 0, "k^", ms=10)
    ax.set_xlim(-8, 8); ax.set_ylim(-0.5, 15); ax.set_aspect("equal")
    ax.set_xlabel("left  ←  metres  →  right", fontsize=8); ax.set_ylabel("metres ahead", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title("bird's-eye view — line = path over frames 0→8 (■ = start)", fontsize=10)

    # Key: number -> object, direction, distance (legend above it)
    key = "\n".join(f"{r.num:>2}. {r.prompt:<12} {r.clock:>2} o'clock  {r.steps:4.1f} steps" + ("  ◀ in path" if r.in_corridor else "")
                    for _, r in o8.iterrows())
    ax_key.text(0, 0.78, key, transform=ax_key.transAxes, fontsize=8, family="monospace", va="top")
    cats = o8.category.unique()
    ax_key.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=CAT_COLORS[c], label=c) for c in cats] +
                      [plt.Line2D([], [], marker="o", ls="", mfc="white", mec="red", mew=2, label="in the path")],
              loc="upper left", fontsize=8, frameon=False, ncol=2)
    fig.suptitle(f"{clip}\nreference (display only): {REMINDER.get(clip, '')}", fontsize=10)
    out = VIZ_DIR / "05_scene" / f"{clip.replace('.frame', '')}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=90, bbox_inches="tight"); plt.show(); plt.close(fig)

for clip in VIZ_CLIPS:
    show_scene(clip)

## 7. Motion (frames 0–8)
- The user's own walking and turning is measured from static landmarks and removed; then each object's velocity, motion type (static / crossing / approaching / moving away / same direction) and time to collision at frame 8.
- Outputs: `ego_motion.parquet`, `motion_frame8.parquet`.

### 7.1. Settings and helpers

In [ ]:
DT = 1 / 3 # seconds between keyframes
STATIC_CATS = {"pole", "sign", "tree_plant", "fence_railing", "barrier_cone", "furniture_stall",
               "street_object", "wall_door", "stairs_step", "curb_edge", "hole_uneven", "water"}  # cannot move by themselves
DEFAULT_WALK = 1.2 # walking speed (m/s) assumed when it can't be measured
STATIC_SPEED = 0.4 # below this (m/s) an object counts as not moving
MAX_MOTION_DIST = 15.0 # motion is only measured for objects closer than this (m)
SPEED_LIMIT = {"pedestrian": 3.0, "animal": 4.0, "two_wheeler": 8.0, "vehicle": 15.0} # faster = measurement noise
HORIZON = 3.0 # look ahead this many seconds for collisions
BODY_ZONE = 1.0 # "reaches": inside the corridor and closer than this (m)

def rigid_2d(A, B):
    """Rotation R and translation t with B ≈ R @ A + t (2-D least squares, Kabsch). A, B: (n, 2)."""
    if len(A) == 1:
        return np.eye(2), B[0] - A[0]
    ca, cb = A.mean(0), B.mean(0)
    U, _, Vt = np.linalg.svd((A - ca).T @ (B - cb))
    R = (U @ Vt).T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1; R = (U @ Vt).T
    return R, cb - R @ ca

def to_h(R, t):
    M = np.eye(3); M[:2, :2] = R; M[:2, 2] = t
    return M

### 7.2. User's walking and turning

In [ ]:
mov = objects[~objects.flat & (objects.conf >= 0.15)].copy()
mov["track_n"] = mov.groupby(["clip_id", "track_id"]).frame.transform("nunique")
ego_rows, to_frame8 = [], {}
for clip in RUN_CLIPS:
    o = mov[mov.clip_id == clip]
    stat = o[o.category.isin(STATIC_CATS) & (o.track_n >= 3) & (o.dist_m < 20) & (o.conf >= 0.3)]
    steps = {}
    for k in range(8):
        a = stat[stat.frame == k].set_index("track_id")[["lateral_m", "forward_m"]]
        b = stat[stat.frame == k + 1].set_index("track_id")[["lateral_m", "forward_m"]]
        common = a.index.intersection(b.index)
        if len(common) == 0:
            steps[k] = None; continue
        A, B = a.loc[common].to_numpy(), b.loc[common].to_numpy()
        R, t = rigid_2d(A, B)
        if len(common) >= 3: # drop landmarks that don't fit (mislabelled movers)
            keep = np.linalg.norm(A @ R.T + t - B, axis=1) < 0.5
            if keep.sum() >= 2:
                R, t = rigid_2d(A[keep], B[keep])
        steps[k] = (R, t, len(common))
    speeds = [-s[1][1] / DT for s in steps.values() if s is not None]
    walk = float(np.clip(np.median(speeds), 0, 2.5)) if speeds else DEFAULT_WALK
    M, Ms = np.eye(3), {8: np.eye(3)} # Ms[k]: frame-k coordinates -> frame-8 coordinates
    for k in range(7, -1, -1):
        R, t, n = steps[k] if steps[k] is not None else (np.eye(2), np.array([0, -walk * DT]), 0)
        M = M @ to_h(R, t); Ms[k] = M.copy()
        yaw = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
        ego_rows.append(dict(clip_id=clip, frame=k, n_static=n, speed=-t[1] / DT, yaw_deg=yaw, measured=n > 0))
    to_frame8[clip] = Ms
ego = pd.DataFrame(ego_rows).sort_values(["clip_id", "frame"])
ego.to_parquet(OUT_DIR / "ego_motion.parquet", index=False)
clip_walk = ego[ego.measured].groupby("clip_id").speed.median().clip(0, 2.5).reindex(RUN_CLIPS).fillna(DEFAULT_WALK)

print(f"own motion measured in {ego.measured.mean():.0%} of frame pairs | median walking speed: {clip_walk.median():.2f} m/s")

### 7.3 Each object's motion and collision prediction

In [ ]:
%%heavy
TW8 = mov[(mov.frame == 8) & (mov.category == "two_wheeler")]
def riding(clip, r):
    """True if a bike/scooter/motorbike is right where this person is (they are riding it)."""
    t = TW8[TW8.clip_id == clip]
    return bool(((t.lateral_m - r.lateral_m).abs() < 0.7).__and__((t.forward_m - r.forward_m).abs() < 1.0).any())

rows = []
for (clip, tid), g in mov.groupby(["clip_id", "track_id"]):
    g = g.sort_values("frame")
    last = g[g.frame == 8]
    if last.empty:
        continue
    r8 = last.iloc[0]
    walk = clip_walk[clip]
    lat, fwd, half_w = r8.lateral_m, r8.forward_m, r8.width_m / 2
    vx = vy = 0.0
    recent = g.frame.to_numpy() >= 3 # last ~2 s
    if r8.category in STATIC_CATS:
        motion = "static" # can't move: any apparent motion is noise
    elif recent.sum() < 3 or r8.dist_m > MAX_MOTION_DIST:
        motion = "unknown" # too few frames, or too far to measure reliably
    else:
        Ms = to_frame8[clip]
        P = np.array([(Ms[k] @ np.array([x, y, 1]))[:2] for k, x, y in zip(g.frame, g.lateral_m, g.forward_m)])
        t = g.frame.to_numpy()[recent] * DT
        vx, vy = np.polyfit(t, P[recent, 0], 1)[0], np.polyfit(t, P[recent, 1], 1)[0]
        speed = np.hypot(vx, vy)
        near_path = abs(lat) - half_w <= CORRIDOR_HALF + 1.0
        limit = SPEED_LIMIT.get(r8.category, 3.0)
        if r8.category == "pedestrian" and riding(clip, r8):
            limit = SPEED_LIMIT["two_wheeler"] # a person on a bike/scooter moves
        if speed > limit: # faster than possible: keep the direction, cap the speed
            vx, vy = vx * limit / speed, vy * limit / speed
            speed = limit
        if speed < STATIC_SPEED:
            motion, vx, vy = "static", 0.0, 0.0
        elif abs(vx) >= 0.5 and (near_path or np.sign(vx) == -np.sign(lat)):
            motion = "crossing" # moving sideways across / into the path
        elif vy < -0.3:
            motion = "approaching" # coming towards 
        elif vy > walk + 0.3:
            motion = "moving_away" # pulling away from 
        else:
            motion = "same_direction" # going in the way
    vrel = np.array([vx, vy - walk])  # its motion relative to the path
    ts = np.arange(0, HORIZON + 1e-9, 0.1)
    path = np.array([lat, fwd]) + np.outer(ts, vrel)
    in_lane = np.abs(path[:, 0]) - half_w <= CORRIDOR_HALF
    hit = in_lane & (path[:, 1] >= 0) & (path[:, 1] <= BODY_ZONE)
    rows.append(dict(clip_id=clip, track_id=tid, category=r8.category, prompt=r8.prompt, conf=r8.conf,
                     dist_m=r8.dist_m, lateral_m=lat, forward_m=fwd, clock=r8.clock, steps=r8.steps,
                     width_m=r8.width_m, in_corridor=r8.in_corridor, n_frames=int(len(g)),
                     vx=float(vx), vy=float(vy), speed=float(np.hypot(vx, vy)), vrel_x=float(vrel[0]), vrel_y=float(vrel[1]),
                     motion=motion, ttc_s=float(ts[hit.argmax()]) if hit.any() else np.nan,
                     enters_corridor_2s=bool((in_lane & (path[:, 1] > 0) & (path[:, 1] <= CORRIDOR_LEN) & (ts <= 2)).any()),
                     walk_speed=walk))
motion8 = pd.DataFrame(rows)
motion8.to_parquet(OUT_DIR / "motion_frame8.parquet", index=False)

print("motion at frame 8:", motion8.motion.value_counts().to_dict())

### 7.4 Visual check: motion

In [ ]:
%%viz
MOTION_STYLE = {"static": "·", "crossing": "crossing", "approaching": "approaching", "moving_away": "moving away",
                "same_direction": "same way as walker", "unknown": "? (unknown)"}
MAP_X, MAP_Y = (-8, 8), (-0.5, 15) # map area (m)
MAX_ARROW = 4.0 # longest arrow drawn (m)

def clip_arrow(start, end):
    """Shorten an arrow so it is at most MAX_ARROW long and ends inside the map."""
    d = end - start
    scale = min(1.0, MAX_ARROW / max(np.linalg.norm(d), 1e-9))
    for i, (lo, hi) in enumerate((MAP_X, MAP_Y)):
        e = start[i] + scale * d[i]
        if e < lo: scale = min(scale, (lo - start[i]) / d[i])
        if e > hi: scale = min(scale, (hi - start[i]) / d[i])
    return start + 0.97 * scale * d

def show_motion(clip):
    o, o8 = scene_objects(clip) # same objects and numbers as the scene card (6.6)
    m = motion8[motion8.clip_id == clip].set_index("track_id")
    o8 = o8[o8.track_id.isin(m.index)]
    W, H = Image.open(frame_path(clip, 8)).size
    img_h = 12 * H / W
    fig = plt.figure(figsize=(12, img_h + 7.5), layout="constrained")
    gs = fig.add_gridspec(2, 2, height_ratios=[img_h, 7.5], width_ratios=[1.3, 1])
    ax_img, ax, ax_key = fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    ax_key.axis("off")

    # Row 1: frame 8, badges outlined orange if moving, red if in the path now or within 2 s
    ax_img.imshow(Image.open(frame_path(clip, 8)))
    for _, r in o8.iterrows():
        mm = m.loc[r.track_id]
        edge = "red" if mm.enters_corridor_2s else ("orange" if mm.motion not in ("static", "unknown") else "white")
        badge(ax_img, r.ground_x, r.ground_y, r.num, CAT_COLORS[r.category], edge)
    ax_img.set_xlim(0, W); ax_img.set_ylim(H, 0); ax_img.axis("off")
    ax_img.set_title("frame 8 — orange ring = moving, red ring = in the path now or within 2 s", fontsize=10)

    # Row 2: map in frame-8 coordinates with the own walking removed
    ax.add_patch(patches.Rectangle((-CORRIDOR_HALF, 0), 2 * CORRIDOR_HALF, CORRIDOR_LEN, color="cyan", alpha=0.25))
    for deg in (-75, -45, -15, 15, 45, 75):
        ax.plot([0, 16 * np.sin(np.radians(deg))], [0, 16 * np.cos(np.radians(deg))], color="lightgrey", lw=0.6)
    for d in (2, 4, 6, 8, 10):
        ax.add_patch(patches.Arc((0, 0), 2 * d, 2 * d, theta1=10, theta2=170, color="lightgrey", lw=0.5))
        ax.text(d * 0.71 + 0.2, d * 0.71, f"{d} m", fontsize=6, color="grey")
    Ms = to_frame8[clip]
    walk = clip_walk[clip]
    ax.annotate("", xy=(0, walk * 2), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="black", lw=1.5))
    for _, r in o8.iterrows():
        mm, c = m.loc[r.track_id], CAT_COLORS[r.category]
        g = mov[(mov.clip_id == clip) & (mov.track_id == r.track_id)].sort_values("frame")
        P = np.array([(Ms[k] @ np.array([x, y, 1]))[:2] for k, x, y in zip(g.frame, g.lateral_m, g.forward_m)])
        if len(P) >= 3: # where it actually went (walking removed)
            P = pd.DataFrame(P).rolling(3, center=True, min_periods=1).median().to_numpy() # smooth out depth jumps
            ax.plot(P[:, 0], P[:, 1], color=c, lw=1.5, alpha=0.6, clip_on=True)
        if mm.motion not in ("static", "unknown"): # where it is heading in the next 2 s
            start = np.array([mm.lateral_m, mm.forward_m])
            if MAP_X[0] < start[0] < MAP_X[1] and MAP_Y[0] < start[1] < MAP_Y[1]:
                end = clip_arrow(start, start + 2 * np.array([mm.vx, mm.vy]))
                ax.annotate("", xy=tuple(end), xytext=tuple(start),
                            arrowprops=dict(arrowstyle="->", color=c, lw=1.5, ls="--"))
        edge = "red" if mm.enters_corridor_2s else ("orange" if mm.motion not in ("static", "unknown") else "white")
        badge(ax, r.lateral_m, r.forward_m, r.num, c, edge)
    ax.plot(0, 0, "k^", ms=10)
    ax.set_xlim(-8, 8); ax.set_ylim(-0.5, 15); ax.set_aspect("equal")
    ax.set_xlabel("left  ←  metres  →  right", fontsize=8); ax.set_ylabel("metres ahead", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title(f"motion — line = path with the walking removed, dashed = next 2 s\n"
                 f"black arrow = walker, {walk:.1f} m/s", fontsize=9)

    key = "\n".join(
        f"{r.num:>2}. {r.prompt:<10} {r.clock:>2}h {r.steps:4.1f}st  {MOTION_STYLE[m.loc[r.track_id].motion]:<15}"
        + (f" {m.loc[r.track_id].speed:.1f} m/s" if m.loc[r.track_id].motion not in ("static", "unknown") else "")
        + (f"  hits in {m.loc[r.track_id].ttc_s:.1f}s" if pd.notna(m.loc[r.track_id].ttc_s) else "")
        + ("  ◀ in the path within 2 s" if m.loc[r.track_id].enters_corridor_2s else "")
        for _, r in o8.iterrows())
    ax_key.text(0, 1, key, transform=ax_key.transAxes, fontsize=7.5, family="monospace", va="top")
    fig.suptitle(f"{clip}\nreference (display only): {REMINDER.get(clip, '')}", fontsize=10)
    out = VIZ_DIR / "06_motion" / f"{clip.replace('.frame', '')}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=90, bbox_inches="tight"); plt.show(); plt.close(fig)

for clip in VIZ_CLIPS:
    show_motion(clip)

## 8. Hazard prioritisation
- Risk score = severity × closeness × position × motion × context × √confidence.
- Surface hazards (water, steps, walls, fences) and unlabelled obstacles come from the segmentation; then the decision (stop / move / slow / caution / clear) and the hint text.
- Outputs: `ranking.parquet`, `hints.jsonl`, `baselines.parquet`.

### 8.1. Settings: severity, likelihood factors and decision thresholds

In [ ]:
# Severity = worst plausible harm to a walker if the encounter happens (likelihood is handled separately)
# Rubric: Q1 energy of the hazard, Q2 fall potential, Q3 rigid contact, Q4 yields/soft; the highest level reached decides
# 4 critical 1.0 | 3 serious 0.8 | 2 moderate 0.6 | 1 minor 0.4
SEVERITY = {
    "vehicle": 1.0,          # 4: moving car/bus/truck/van/tricycle - high-energy impact (parked: see SEVERITY_PARKED)
    "water": 1.0,            # 4: river/canal/lake/fountain - fall into water, drowning (puddle: see SEVERITY_PROMPT)
    "two_wheeler": 0.8,      # 3: moving bicycle/motorcycle/scooter - moderate-speed impact
    "stairs_step": 0.8,      # 3: missed step -> fall (rated for going down)
    "curb_edge": 0.8,        # 3: unexpected step down -> trip and fall
    "hole_uneven": 0.8,      # 3: pothole -> foot caught, fall (speed bump: see SEVERITY_PROMPT)
    "pole": 0.6,             # 2: rigid, narrow, head-level collisions
    "sign": 0.6,             # 2: rigid edges, head height or low boards
    "tree_plant": 0.6,       # 2: rigid trunk, head-level branches (bush/pot share the category)
    "fence_railing": 0.6,    # 2: rigid, usually alongside the path
    "wall_door": 0.6,        # 2: rigid; doors open into the path; glass is hard to see
    "barrier_cone": 0.6,     # 2: low rigid bollards/barriers - knock or trip
    "furniture_stall": 0.6,  # 2: benches, tables, stalls, carts - rigid, shin to waist height
    "street_object": 0.6,    # 2: bins, boxes, vending machines, statues, suitcases - rigid knock
    "obstacle": 0.6,         # 2: unlabelled thing standing up from the ground - unknown rigid object
    "pedestrian": 0.4,       # 1: walking speed, soft, steps aside
    "animal": 0.4,           # 1: usually moves away by itself
}
SEVERITY_PARKED = {"vehicle": 0.6, "two_wheeler": 0.6}     # not moving: no impact energy, just a rigid object
SEVERITY_PROMPT = {"puddle": 0.6, "speed bump": 0.6}       # milder members of their category (slip / small trip)
ON_ROAD_VEHICLE = 1.3        # likelihood: on the road a moving vehicle is more likely to reach walker
FENCED_WATER = 0.3           # likelihood: a fence between walker and the water makes falling in unlikely
ALONGSIDE = 0.5              # likelihood: a wall/fence running alongside walker is only hit if walker drift into it
DIST_SCALE = 8.0             # closeness = exp(-distance / 8 m): 0.65 at 3.5 m, 0.42 at 7 m, 0.29 at 10 m
SIDE_GAP = [0, 0.5, 1.5, 3.0, 5.0]           # metres outside the corridor edge ...
SIDE_W = [1.0, 0.7, 0.4, 0.15, 0.0]          # ... and the weight at that gap
PERSONAL_ZONE = 2.0          # a moving thing closer than this counts as fully in the way
MIN_CONF, MIN_CONF_NEAR, NEAR_CONF_DIST = 0.25, 0.15, 6.0   # low-confidence detections kept only if within 6 m
MAX_DIST = 20.0
THIN_WIDTH = {"tree_plant": 0.5, "pole": 0.3, "sign": 0.3}   # widest a trunk / post really is (m)
MOTION_FACTOR = {"static": 1.0, "unknown": 1.0, "same_direction": 1.0, "moving_away": 0.7,
                 "approaching": 1.2, "crossing": 1.2}
MOVING = {"crossing", "approaching", "same_direction", "moving_away"}

# decision levels
STOP_TOUCH = 1.0             # anything in the path closer than this -> stop
STOP_MOVING_DIST, STOP_TTC = 3.0, 1.5   # a moving object reaching the path within 1.5 s and 3 m -> stop
MOVE_DIST = 5.0              # a fixed obstacle in the path within 5 m (~7 steps, ~4 s) -> move around it
SLOW_NEAR = 3.0              # someone/something that can move, in the path within 3 m -> slow down
SLOW_ENTER = 6.0             # ... or moving towards/across the path, entering it within 2 s, within 6 m -> slow down
SLOW_ROAD = 8.0              # on the road: a moving vehicle within 8 m -> slow down
CAUTION_PATH, CAUTION_SIDE, SIDE_LANE = 12.0, 8.0, 1.5  # caution: straight ahead <= 12 m (~8 s), or beside (<= 1.5 m out) <= 8 m

# label-free obstacle check
OBST_FWD, OBST_SIDE = 4.0, 1.5        # look up to 4 m ahead, up to 1.5 m beside the corridor
OBST_HEIGHT = (0.2, 2.0)              # pixels standing this high above the ground (m)
OBST_MIN_FRAC = 0.004                 # a patch must cover at least 0.4% of the image

# getting past a fixed obstacle: where across the width ahead can the body fit?
BIN = 0.1                             # lateral resolution of the free-space check (m)
LANE_MAX_SHIFT = 3.0                  # look up to 3 m left and right
LANE_MARGIN = 1.5                     # the way must stay free until 1.5 m past the obstacle
BODY_HALF = 0.3                       # half the shoulder width (m)
TIGHT_GAP, COMFY_GAP = 0.1, 0.3       # clearance each side: "narrow" if only 0.1 m, comfortable with 0.3 m
LANE_BLOCK_PX = 3                     # this many blocking pixels at one lateral position = blocked
LANE_MIN_WALK = 0.5                   # at least half the ground there must be walkable
LANE_WALK = [1, 3]                    # walkable: pavement, terrain (road only if walker's already on it, or seg mixes them)
PAVEMENT_PREF = 0.2                   # prefer the side with this much more pavement
WALL_SIDE_GAP = 1.0                   # a wall/fence this close outside the path (m) is mentioned
FOOT_MIN_COLS = 3                     # a wall/fence must touch the ground over at least 3 image columns
STRAIGHT_TOL = 0.2                    # drifting up to 0.2 m sideways still counts as "keep straight"
STOP_BLOCKED_DIST = 2.0               # no way around and the obstacle within 2 m: stop rather than slow
BLOCKED_GROUND, BLOCKED_UNBOXED, BLOCKED_DEPTH = 0.1, 0.7, 3.0   # view blocked: <10% ground in front, >70% unboxed, <3 m
NARROW_AHEAD = 4.0                    # check the next 4 m for a narrow passage
MIXED_SHARE = 0.4                     # road and pavement count as mixed if 40% of positions are a road/pavement blend

def side_weight(gap):
    return np.interp(gap, SIDE_GAP, SIDE_W)

### 8.2. Surface hazards, walls and fences, unlabelled obstacles and context

In [ ]:
from scipy import ndimage

def ground_coords(clip, k=8): # Every pixel of the 320-px grid placed on the ground plane (lateral, forward in m), NaN above the horizon
    groups = np.load(CACHE_DIR / "seg" / f"{clip}.npz")["groups"][k]
    h, w = groups.shape
    gr = GEO.loc[(clip, k)]
    a = np.array([gr.ax, gr.ay, gr.az]); fwd, right = ground_axes(a); foot = -CAM_HEIGHT * a
    f, cx, cy = intrinsics(w, h)
    ys, xs = np.mgrid[0:h, 0:w]
    rays = np.stack([(xs - cx) / f, (ys - cy) / f, np.ones_like(xs, float)], -1)
    denom = rays @ a
    t = np.where(denom < -1e-6, -CAM_HEIGHT / np.where(denom < -1e-6, denom, -1), np.nan)
    P = rays * t[..., None] - foot
    return groups, P @ right, P @ fwd, (a, fwd, right, foot, rays, gr)

def to_image(clip, ys, xs, shape): # Grid pixel -> full-image pixel (for drawing)
    W, H = Image.open(frame_path(clip, 8)).size
    return float(np.median(xs)) * W / shape[1], float(np.max(ys)) * H / shape[0]

def make_hazard(clip, mask, lat, fwd, groups, category, prompt, in_path):
    ys, xs = np.where(mask)
    d = float(np.nanpercentile(np.hypot(lat[mask], fwd[mask]), 10))
    lm, fm = float(np.nanmedian(lat[mask])), float(np.nanmedian(fwd[mask]))
    gx, gy = to_image(clip, ys, xs, groups.shape)
    return dict(category=category, prompt=prompt, dist_m=d, lateral_m=lm, forward_m=fm,
                clock=int(round(np.degrees(np.arctan2(lm, fm)) / 30)) % 12 or 12, steps=d / STEP_M,
                in_corridor=bool(in_path), width_m=float(np.nanpercentile(lat[mask], 90) - np.nanpercentile(lat[mask], 10)),
                conf=0.5 if category == "obstacle" else 1.0, motion="static", ttc_s=np.nan, enters_corridor_2s=bool(in_path),
                ground_x=gx, ground_y=gy, source=category if category == "obstacle" else "surface")

def footprint(groups, gids): 
    # Where a wall/fence meets the ground, one pixel per image column
    # ground = the ground pixel just below it
    # cut = its bottom-row pixel when it runs off the bottom of the image
    m = np.isin(groups, gids)
    h = m.shape[0]
    low = h - 1 - np.argmax(m[::-1], 0)
    cols = np.where(m.any(0))[0]
    ground, cut = np.zeros_like(m), np.zeros_like(m)
    ok = low[cols] < h - 1
    ground[low[cols][ok] + 1, cols[ok]] = True
    cut[h - 1, cols[~ok]] = True
    return ground, cut

def surface_hazards(clip, boxes):
    groups, lat, fwd, (a, f_ax, right, foot, rays, gr) = ground_coords(clip)
    h, w = groups.shape
    ahead = (fwd > 0.3) & (fwd <= CORRIDOR_LEN)
    gap = np.abs(lat) - CORRIDOR_HALF
    in_path = ahead & (gap <= 0)
    side = {"left": ahead & (lat < -CORRIDOR_HALF) & (gap <= 2.0), "right": ahead & (lat > CORRIDOR_HALF) & (gap <= 2.0)}
    out, flags = [], {}
    # road vs pavement: cleanly separated, or blended together by the segmentation?
    zone = (np.abs(lat) <= 3) & (fwd > 0.3) & (fwd <= 6) & np.isin(groups, [1, 2])
    edges = np.arange(-3, 3.01, 0.25)
    n_all = np.histogram(lat[zone], edges)[0]
    n_road = np.histogram(lat[zone & (groups == 2)], edges)[0]
    share = n_road[n_all >= 10] / n_all[n_all >= 10]
    road_frac = (groups[zone] == 2).mean() if zone.any() else 0.0
    flags["mixed_ground"] = bool(0.15 <= road_frac <= 0.85 and len(share)
                                 and ((share >= 0.25) & (share <= 0.75)).mean() >= MIXED_SHARE)
    near = in_path & (fwd <= 5)
    flags["on_road"] = bool((groups[near] == 2).mean() > 0.5) if near.any() and not flags["mixed_ground"] else False
    # 3-D position and height above the ground of every pixel (depth, scaled to metres)
    depth = np.load(CACHE_DIR / "depth" / f"{clip}.npz")["depth"][8].astype(np.float32) * gr.scale
    P = rays * depth[..., None]
    height = P @ a + CAM_HEIGHT
    Q = P - foot
    plat, pfwd = Q @ right, Q @ f_ax
    # walls/fences: ground contact from the plane; if cut off at the bottom of the image, from depth instead
    wlat, wfwd, fp = lat.copy(), fwd.copy(), {}
    for gid in (6, 7):
        ground, cut = footprint(groups, [gid])
        wlat[cut], wfwd[cut] = plat[cut], pfwd[cut]
        fp[gid] = (ground | cut) & (wfwd > 0)
    # water and steps lie on the ground: in the path or beside it (up to 2 m out)
    for gid, cat, name in ((4, "water", "water"), (5, "stairs_step", "steps")):
        for where, mask in [("path", in_path), ("left", side["left"]), ("right", side["right"])]:
            hit = mask & (groups == gid)
            if hit.sum() >= 15:
                hz = make_hazard(clip, hit, lat, fwd, groups, cat, name, where == "path")
                hz["fenced"] = where != "path" and (fp[6] & side[where]).sum() >= FOOT_MIN_COLS
                out.append(hz)
    # fence / wall, placed where they meet the ground: across Grid pixel -> full-image pixel path (<= 5 m) = in the way
    # running alongside walker, or right beside the path (<= 1 m out, <= 5 m ahead) = beside walker
    for gid, cat, name in ((6, "fence_railing", "fence"), (7, "wall_door", "wall")):
        wgap = np.abs(wlat) - CORRIDOR_HALF
        hit = fp[gid] & (wgap <= 0) & (wfwd <= MOVE_DIST)
        if hit.sum() >= FOOT_MIN_COLS:
            across = np.ptp(np.percentile(wlat[hit], [10, 90]))
            along = np.ptp(np.percentile(wfwd[hit], [10, 90]))
            if across >= 0.6 or across >= 0.5 * along:
                out.append(make_hazard(clip, hit, wlat, wfwd, groups, cat, name, True))
                continue
        for s, sgn in (("left", -1), ("right", 1)):
            hit = fp[gid] & (wlat * sgn > 0) & (wgap <= WALL_SIDE_GAP) & (wfwd <= CAUTION_SIDE)
            if hit.sum() >= FOOT_MIN_COLS:
                out.append(make_hazard(clip, hit, wlat, wfwd, groups, cat, name, False))
    surf = ndimage.binary_dilation(np.isin(groups, [4, 5, 6, 7]), iterations=2)   # water/steps/fence/wall (+ their edges)
    standing = ((height >= OBST_HEIGHT[0]) & (height <= OBST_HEIGHT[1]) & ~np.isin(groups, WALKABLE) & ~surf
                if gr.plane_ok else np.zeros_like(groups, bool))
    # unlabelled obstacles: pixels that stand up from the ground near the path and no detector box covers
    covered = np.zeros_like(groups, bool)
    W, H = Image.open(frame_path(clip, 8)).size
    for _, b in boxes.iterrows():
        covered[int(b.y1 / H * h): int(np.ceil(b.y2 / H * h)), int(b.x1 / W * w): int(np.ceil(b.x2 / W * w))] = True
    # right in front of the feet: is there any ground, or only something solid that no detector box explains?
    bc = np.zeros_like(groups, bool)
    bc[int(h * 0.67):, w // 3: 2 * w // 3] = True                  # bottom-centre of the image
    flags["front_ground"] = float(np.isin(groups[bc], [1, 2, 3]).mean())
    flags["front_unboxed"] = float((~covered[bc]).mean())
    flags["front_depth_m"] = float(np.nanmedian(depth[bc]))
    flags["view_blocked"] = bool(flags["front_ground"] < BLOCKED_GROUND and flags["front_unboxed"] > BLOCKED_UNBOXED
                                 and flags["front_depth_m"] < BLOCKED_DEPTH)
    if gr.plane_ok:
        cand = (pfwd > 0.3) & (pfwd <= OBST_FWD) & (np.abs(plat) - CORRIDOR_HALF <= OBST_SIDE) & standing & ~covered
        lab, n = ndimage.label(cand)
        for i in range(1, n + 1):
            blob = lab == i
            if blob.mean() >= OBST_MIN_FRAC:
                hz = make_hazard(clip, blob, plat, pfwd, groups, "obstacle", "obstacle",
                                 bool((np.abs(plat[blob]) - CORRIDOR_HALF <= 0).mean() > 0.3))
                hz["height_m"] = float(np.nanmedian(height[blob]))
                out.append(hz)
    # what decide() needs to find where walker fit: blocking points and ground pixels near the path (in metres)
    keep_l = lambda L, F: (np.abs(L) <= 4) & (F > 0.3) & (F <= CORRIDOR_LEN)
    scene = dict(blockers=[], ground=None, mixed=flags["mixed_ground"])
    for mask, L, F, name, n_min in ((fp[6], wlat, wfwd, "fence", FOOT_MIN_COLS), (fp[7], wlat, wfwd, "wall", FOOT_MIN_COLS),
                                     (groups == 4, lat, fwd, "water", LANE_BLOCK_PX), (groups == 5, lat, fwd, "steps", LANE_BLOCK_PX),
                                     (standing & ~covered, plat, pfwd, "obstacle", LANE_BLOCK_PX)):
        m = mask & keep_l(L, F)
        scene["blockers"].append((L[m], F[m], name, n_min))
    m = keep_l(lat, fwd) & np.isin(groups, [1, 2, 3, 4, 5]) # real ground surfaces only (not walls)
    scene["ground"] = (lat[m], fwd[m], groups[m])
    return out, flags, scene

### 8.3. Score, find a way past, decide and write the hint

In [ ]:
def score_candidates(cands, flags):
    #risk score = severity x likelihood (closeness x position x motion x context) x sqrt(confidence)
    c = cands.copy()
    parked = c.category.isin(list(SEVERITY_PARKED)) & (c.motion == "static")
    sev = c.category.map(SEVERITY).fillna(0.6)
    sev = sev.where(~parked, c.category.map(SEVERITY_PARKED))
    sev = c.prompt.map(SEVERITY_PROMPT).fillna(sev)
    fixed = c.category.isin(STATIC_CATS | {"obstacle"}) | parked
    context = np.where(flags["on_road"] & (c.category == "vehicle") & ~parked, ON_ROAD_VEHICLE, 1.0)
    context = context * np.where((c.category == "water") & c.fenced.fillna(False).astype(bool), FENCED_WATER, 1.0)
    context = context * np.where(c.category.isin(["wall_door", "fence_railing"]) & (c.source == "surface") & ~c.in_corridor,
                                 ALONGSIDE, 1.0)
    c["gap_m"] = (c.lateral_m.abs() - c.width_m / 2 - CORRIDOR_HALF).clip(lower=0)
    path = np.where(c.in_corridor | (~fixed & (c.dist_m <= PERSONAL_ZONE)), 1.0, side_weight(c.gap_m))
    motion = c.motion.map(MOTION_FACTOR).fillna(1.0) * np.where(c.enters_corridor_2s & ~c.in_corridor & c.motion.isin(MOVING), 1.5, 1.0)
    c["parked"] = parked
    c["s_severity"], c["s_close"], c["s_path"], c["s_motion"], c["s_context"] = sev.to_numpy(), np.exp(-c.dist_m / DIST_SCALE), path, motion, context
    c["score"] = c.s_severity * c.s_close * c.s_path * c.s_motion * c.s_context * np.sqrt(c.conf)
    return c

def free_space(scene, c, need, walk_groups, skip=None):
    # Lateral positions -3..3 m (every 0.1 m)
    # blocked = something stands there, or the ground is not walkable
    # anywhere between 0.3 m and need metres ahead. Also returns solid things, solid things except `skip`,
    # and the pavement share at each position."""
    xs = np.round(np.arange(-LANE_MAX_SHIFT, LANE_MAX_SHIFT + 1e-9, BIN), 2)
    edges = np.append(xs - BIN / 2, xs[-1] + BIN / 2)
    blocked = np.zeros(len(xs), bool)
    for L, F, name, n_min in scene["blockers"]:
        m = F <= need
        if m.sum() >= n_min: # enough of it overall to be real
            blocked |= np.histogram(L[m], edges)[0] >= (1 if name in ("fence", "wall") else LANE_BLOCK_PX)
    others = blocked.copy() # solid things, except the obstacle walker are stepping around
    near = c[(c.forward_m > 0) & (c.forward_m <= need) & (c.motion != "moving_away")]
    for j, o in near.iterrows():
        span = (xs >= o.lateral_m - o.width_m / 2) & (xs <= o.lateral_m + o.width_m / 2)
        blocked |= span
        if j != skip:
            others |= span
    solid = blocked.copy() # things standing there (without the ground check)
    L, F, G = scene["ground"]
    m = F <= need
    tot = np.histogram(L[m], edges)[0]
    walk = np.histogram(L[m & np.isin(G, walk_groups)], edges)[0]
    pave = np.histogram(L[m & np.isin(G, [1, 2] if scene["mixed"] else [1])], edges)[0]
    blocked |= (tot > 0) & (walk < LANE_MIN_WALK * tot) # ground seen there, but not walkable (unseen = unknown, not blocked)
    return xs, blocked, solid, others, pave / np.maximum(tot, 1)

def reachable(others, xs, sgn, span):
    # Positions walker can reach by stepping sideways (sgn -1 left, +1 right) without crossing anything solid, apart from the obstacle walker is stepping around (its lateral span, +-0.1 m)
    reach = np.zeros(len(xs), bool)
    for j in (np.flatnonzero(xs >= 0) if sgn > 0 else np.flatnonzero(xs <= 0)[::-1]):
        if others[j] and not (span[0] - 0.1 <= xs[j] <= span[1] + 0.1):
            break
        reach[j] = True
    return reach

def body_fits(blocked, gap):
    k = int(round((BODY_HALF + gap) / BIN))
    return ~ndimage.binary_dilation(blocked, iterations=k, border_value=1)

def runs(mask, xs):
    # Continuous stretches of True as [start, end] in metres
    idx = np.flatnonzero(np.diff(np.r_[0, mask.astype(int), 0]))
    return [[float(xs[s]), float(xs[e - 1])] for s, e in zip(idx[::2], idx[1::2])]

def first_in_line(scene, c, x, start, skip):
    # The first thing further along the new line (x +- body), beyond `start` metres: (name, distance) or None
    lo, hi, best = x - BODY_HALF - TIGHT_GAP, x + BODY_HALF + TIGHT_GAP, None
    for L, F, name, n_min in scene["blockers"]:
        f = np.sort(F[(L >= lo) & (L <= hi) & (F > start)])
        if len(f) >= n_min and (best is None or f[n_min - 1] < best[1]):
            best = (name, float(f[n_min - 1]))
    o = c[(c.lateral_m + c.width_m / 2 >= lo) & (c.lateral_m - c.width_m / 2 <= hi) & (c.forward_m > start)
          & (c.forward_m <= CORRIDOR_LEN) & (c.motion != "moving_away") & (c.index != skip)]
    if len(o) and (best is None or o.forward_m.min() < best[1]):
        best = (o.loc[o.forward_m.idxmin()].prompt, float(o.forward_m.min()))
    return best

def decide(c, flags, scene):
    # Returns (level, move_clock, index of the object that triggered it)
    # First matching rule wins
    flags.update(passage="", move_then="", aim_m=np.nan, check_until_m=np.nan, free_wide=[], free_narrow=[])
    if flags["view_blocked"]:                              # no ground in front of the feet, only something solid
        flags["passage"] = "blocked_view"
        near = c[c.forward_m > 0] if len(c) else c
        return "stop", None, near.dist_m.idxmin() if len(near) else None
    if not len(c):
        return "clear", None, None
    fixed = c.category.isin(STATIC_CATS | {"obstacle"}) | c.parked # will not get out of the way
    moving = ~fixed # people (even standing), animals, moving/unknown vehicles
    in_path = c.in_corridor | (moving & (c.dist_m <= PERSONAL_ZONE) & (c.gap_m <= 0.3))  # walls walker walk past are not "in path"
    beside = ~in_path & (c.gap_m <= SIDE_LANE)
    pick = lambda m: c[m].dist_m.idxmin()
    m = (c.motion.isin(MOVING) & (c.ttc_s.fillna(99) <= STOP_TTC) & (c.dist_m <= STOP_MOVING_DIST)) | (in_path & (c.dist_m <= STOP_TOUCH))
    if m.any():
        return "stop", None, pick(m)
    m = in_path & fixed & (c.dist_m <= MOVE_DIST)
    if m.any():
        i = pick(m); r = c.loc[i]
        need = r.forward_m + LANE_MARGIN  # the way must be free until just past the obstacle
        walk_groups = LANE_WALK + ([2] if flags["on_road"] or scene["mixed"] else [])
        xs, blocked, solid, others, pave = free_space(scene, c, need, walk_groups, skip=i)
        wide, narrow = body_fits(blocked, COMFY_GAP), body_fits(blocked, TIGHT_GAP)
        span = (r.lateral_m - r.width_m / 2, r.lateral_m + r.width_m / 2)
        if r.source == "surface":  # a wall/fence: the whole connected stretch of it is the obstacle
            lab, _ = ndimage.label(ndimage.binary_closing(solid, iterations=2))   # bridge gaps up to 0.2 m
            ids = set(lab[(xs >= span[0]) & (xs <= span[1])]) - {0}
            if ids:
                run = xs[np.isin(lab, list(ids))]
                span = (min(span[0], run.min()), max(span[1], run.max()))
        flags.update(check_until_m=float(need), free_wide=runs(wide, xs), free_narrow=runs(narrow & ~wide, xs))
        straight = np.abs(xs) <= STRAIGHT_TOL  # a small sideways drift still counts as going straight
        for fits, kind, level in ((wide, "straight", "caution"), (narrow, "narrow_straight", "slow")):
            if (fits & straight).any():
                j = np.flatnonzero(fits & straight)[np.argmin(np.abs(xs[fits & straight]))]
                flags.update(passage=kind, aim_m=float(xs[j]))
                return level, None, i
        for fits, kind in ((wide, "wide"), (narrow, "narrow")):
            options = {}
            for s, sgn in (("left", -1), ("right", 1)):
                cand = np.flatnonzero(fits & (xs * sgn > 0) & reachable(others, xs, sgn, span))
                if len(cand):
                    j = cand[np.argmin(np.abs(xs[cand]))] # the smallest step aside on this side
                    win = np.abs(xs - xs[j]) <= BODY_HALF
                    options[s] = (float(xs[j]), float(pave[win].mean()))
            if options:
                break
        if not options: # nowhere to fit: don't steer into something
            flags["passage"] = "none"
            return ("stop" if r.dist_m <= STOP_BLOCKED_DIST else "slow"), None, i
        if len(options) == 2: # both sides: more pavement wins, otherwise the smaller step
            dp = options["left"][1] - options["right"][1]
            side = ("left" if dp > 0 else "right") if abs(dp) >= PAVEMENT_PREF else \
                   min(options, key=lambda s: abs(options[s][0]))
        else:
            side = next(iter(options))
        x = options[side][0]
        ang = np.degrees(np.arctan2(abs(x), max(r.forward_m - 0.5, 0.3))) # aim there, half a metre before the obstacle
        hours = int(np.clip(round(ang / 30), 1, 3))
        clock = 12 - hours if side == "left" else hours
        flags.update(passage=kind, aim_m=x)
        nxt = first_in_line(scene, c, x, need, i)
        if nxt:
            flags["move_then"] = f", then mind the {nxt[0]} in about {max(1, round(nxt[1] / STEP_M))} steps"
        return ("move" if kind == "wide" else "slow"), clock, i
    # nothing to step around, but are walls/things on both sides squeezing the path?
    xs, _, solid, _, _ = free_space(scene, c[fixed], NARROW_AHEAD, [1, 2, 3, 4, 5, 6, 7, 0])
    near_me = np.abs(xs) <= 0.5
    if body_fits(solid, TIGHT_GAP)[np.argmin(np.abs(xs))] and not body_fits(solid, COMFY_GAP)[near_me].any():
        flags.update(passage="narrow_straight", check_until_m=float(NARROW_AHEAD),
                     free_narrow=runs(body_fits(solid, TIGHT_GAP), xs))
        side = c[fixed & (c.dist_m <= NARROW_AHEAD + 1)]
        return "slow", None, side.dist_m.idxmin() if len(side) else None
    m = ((in_path & moving & (c.dist_m <= SLOW_NEAR) & (c.motion != "moving_away"))
         | (moving & c.motion.isin(["crossing", "approaching"]) & c.enters_corridor_2s & (c.dist_m <= SLOW_ENTER)))
    if m.any():
        return "slow", None, pick(m)
    m = flags["on_road"] & (c.category == "vehicle") & moving & (c.dist_m <= SLOW_ROAD)
    if m.any():
        return "slow", None, pick(m)
    in_line = (c.forward_m > 0) & (c.lateral_m.abs() - c.width_m / 2 <= CORRIDOR_HALF)   # straight ahead, any distance
    m = (in_line & (c.dist_m <= CAUTION_PATH)) | (beside & (c.dist_m <= CAUTION_SIDE))
    if m.any():
        return "caution", None, c[m].score.idxmax()
    return "clear", None, None

def hint_text(ranked, level, move_clock, flags, cause="", k=3):
    if level == "clear":
        return f"Path ahead clear for about {int(CORRIDOR_LEN / STEP_M)} steps. Action: continue straight."
    lines = []
    for i, r in ranked.head(k).iterrows():
        where = ("in the path" if r.in_corridor else "further ahead" if r.gap_m == 0 and r.forward_m > CORRIDOR_LEN else
                 "beside the path" if r.gap_m <= SIDE_LANE else "to the side")
        mot = "" if r.motion in ("static", "unknown") else f" | {r.motion.replace('_', ' ')}"
        name = r.prompt + (" (parked)" if r.parked else "")
        lines.append(f"{i + 1}) {name} | {r.clock} o'clock | {max(1, round(r.steps))} steps | {where}{mot}")
    p = flags["passage"]
    lean = ("" if pd.isna(flags["aim_m"]) or abs(flags["aim_m"]) < 0.05 else
            f", slightly to the {'left' if flags['aim_m'] < 0 else 'right'}")
    the = f" the {cause}" if cause else " it"
    if level == "stop" and p == "blocked_view":
        act = "stop, something is right in front of you"
    elif level == "stop" and p == "none":
        act = f"stop, there is no clear way around{the}"
    elif level == "stop":
        act = f"stop and wait for{the}"
    elif level == "move":
        act = f"move towards {move_clock} o'clock to pass{the}{flags['move_then']}"
    elif level == "slow" and p == "narrow":
        act = f"slow down and move towards {move_clock} o'clock to pass{the}, the gap is narrow{flags['move_then']}"
    elif level == "slow" and p == "narrow_straight":
        act = f"slow down and keep straight{lean}, the path is narrow"
    elif level == "slow" and p == "none":
        act = f"slow down, no clear way around{the}"
    elif level == "slow":
        act = f"slow down for{the}" if cause else "slow down"
    elif p == "straight":
        act = f"keep going{lean}, be careful"
    else:
        act = "keep going, be careful"
    ctx = [n for n, on in (("on the road", flags["on_road"]), ("crowded", flags["crowded"])) if on]
    return "\n".join(lines + [f"Action: {act}."] + ([f"Context: {', '.join(ctx)}."] if ctx else []))

### 8.4. Run for every clip and save

In [ ]:
obj8 = objects[objects.frame == 8][["clip_id", "track_id", "flat", "ground_x", "ground_y"]]
cand_all = motion8.merge(obj8, on=["clip_id", "track_id"])
keep = (cand_all.conf >= MIN_CONF) | ((cand_all.conf >= MIN_CONF_NEAR) & (cand_all.dist_m <= NEAR_CONF_DIST))
cand_all = cand_all[~cand_all.flat & keep & (cand_all.dist_m <= MAX_DIST) & (cand_all.forward_m > 0)].copy()
# a tree's or sign's box is as wide as its canopy/board, but only the trunk/post stands in the way
cand_all["width_m"] = np.minimum(cand_all.width_m, cand_all.category.map(THIN_WIDTH).fillna(np.inf))
cand_all["in_corridor"] = ((cand_all.forward_m > 0) & (cand_all.forward_m <= CORRIDOR_LEN)
                           & (cand_all.lateral_m.abs() - cand_all.width_m / 2 <= CORRIDOR_HALF))
boxes8 = det[(det.frame == 8) & (det.conf >= MIN_CONF_NEAR)]

rank_rows, hint_rows, base_rows = [], [], []
rng_base = np.random.default_rng(SEED)
for clip in RUN_CLIPS:
    cands = cand_all[cand_all.clip_id == clip].assign(source="object", fenced=False)
    surf_c, flags, scene = surface_hazards(clip, boxes8[boxes8.clip_id == clip])
    flags["crowded"] = bool(((cands.category == "pedestrian") & (cands.dist_m <= 8)).sum() >= 6)
    if surf_c:
        cands = pd.concat([cands, pd.DataFrame(surf_c)], ignore_index=True)
    cands["fenced"] = cands.fenced.eq(True)
    scored = score_candidates(cands.reset_index(drop=True), flags) if len(cands) else cands
    level, move_clock, trig = decide(scored, flags, scene)
    if len(scored):
        scored["trigger"] = scored.index == trig # what caused the action (named in the action line)
        ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)
        ranked = ranked.assign(clip_id=clip, rank=np.arange(1, len(ranked) + 1))
    else:
        ranked = scored
    rank_rows.append(ranked)
    top = ranked.iloc[0] if len(ranked) and level != "clear" else None
    r1 = ranked.iloc[0] if len(ranked) else None # highest score, whatever the decision
    tr = ranked[ranked.trigger].iloc[0] if len(ranked) and ranked.trigger.any() else None
    cause = "" if tr is None else ("parked " if tr.parked else "") + tr.prompt
    hint_rows.append(dict(clip_id=clip, level=level, clear=level == "clear", action=level, action_clock=move_clock,
                          top_category=None if top is None else top.category, top_prompt=None if top is None else top.prompt,
                          top_clock=None if top is None else int(top.clock), top_steps=None if top is None else float(top.steps),
                          top_in_path=None if top is None else bool(top.in_corridor), top_source=None if top is None else top.source,
                          rank1_category=None if r1 is None else r1.category, rank1_clock=None if r1 is None else int(r1.clock),
                          trigger_category=None if tr is None else tr.category, cause=cause,
                          n_candidates=len(ranked), **flags, hint=hint_text(ranked, level, move_clock, flags, cause)))
    # simple baselines for comparison: which single object would each rule pick?
    objs = cands[cands.source == "object"]
    for name, pick in [("nearest", lambda d: d.dist_m.idxmin()), ("largest", lambda d: (d.width_m / d.dist_m).idxmax()),
                       ("most_confident", lambda d: d.conf.idxmax()), ("random", lambda d: rng_base.choice(d.index))]:
        b = objs.loc[pick(objs)] if len(objs) else None
        base_rows.append(dict(clip_id=clip, baseline=name, clear=b is None,
                              top_category=None if b is None else b.category, top_clock=None if b is None else int(b.clock)))

ranking = pd.concat([r for r in rank_rows if len(r)], ignore_index=True)
hints = pd.DataFrame(hint_rows)
baselines = pd.DataFrame(base_rows)
ranking.to_parquet(OUT_DIR / "ranking.parquet", index=False)
hints.to_json(OUT_DIR / "hints.jsonl", orient="records", lines=True, force_ascii=False)
baselines.to_parquet(OUT_DIR / "baselines.parquet", index=False)

print("decision levels:", hints.level.value_counts().to_dict())
print("top hazard:", hints.top_category.value_counts().head(8).to_dict())

### 8.5. Visual check: priority card

In [ ]:
%%viz
import textwrap
RANK_COLORS = {1: "red", 2: "darkorange", 3: "gold"}
LEVEL_STYLE = {"stop": ("STOP", "red"), "move": ("MOVE", "orangered"), "slow": ("SLOW DOWN", "darkorange"),
               "caution": ("CAUTION — keep going", "goldenrod"), "clear": ("CLEAR — continue", "green")}

def banner_text(h):
    text, color = LEVEL_STYLE[h.level]
    if pd.notna(h.action_clock):
        text += f" → {int(h.action_clock)} o'clock"
    return text, color

def show_priority(clip):
    r = ranking[ranking.clip_id == clip]
    h = hints[hints.clip_id == clip].iloc[0]
    top3 = r.head(3) if h.level != "clear" else r.head(0)
    W, H = Image.open(frame_path(clip, 8)).size
    img_h = 12 * H / W
    fig = plt.figure(figsize=(12, img_h + 7.5), layout="constrained")
    gs = fig.add_gridspec(2, 2, height_ratios=[img_h, 7.5], width_ratios=[1.3, 1])
    ax_img, ax, ax_key = fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    ax_key.axis("off")

    # Row 1: frame 8 with the corridor; top-3 hazards as rank badges, everything else as small grey dots
    ax_img.imshow(Image.open(frame_path(clip, 8)))
    gr = GEO.loc[(clip, 8)]
    a = np.array([gr.ax, gr.ay, gr.az]); fwd, right = ground_axes(a); foot = -CAM_HEIGHT * a
    f, cx, cy = intrinsics(W, H)
    for side in (-CORRIDOR_HALF, CORRIDOR_HALF):
        pts = [project(foot + side * right + z * fwd, f, cx, cy) for z in np.linspace(1, CORRIDOR_LEN, 15)]
        ax_img.plot(*zip(*pts), color="cyan", lw=1.5, ls="--")
    ax_img.scatter(r.ground_x, r.ground_y, s=12, color="lightgrey", edgecolor="black", lw=0.3, zorder=3, clip_on=True)
    for _, t in r[r.source == "obstacle"].iterrows():                  # unlabelled obstacles: dashed marker
        ax_img.scatter(t.ground_x, t.ground_y, s=260, facecolor="none", edgecolor="magenta", lw=1.5, ls="--", zorder=3, clip_on=True)
    for _, t in top3.iterrows():
        badge(ax_img, t.ground_x, t.ground_y, t["rank"], RANK_COLORS[t["rank"]], "white")
    ax_img.set_xlim(0, W); ax_img.set_ylim(H, 0); ax_img.axis("off")
    text, color = banner_text(h)
    ax_img.text(0.01, 0.97, text, transform=ax_img.transAxes, fontsize=13, fontweight="bold", va="top",
                color="white", bbox=dict(facecolor=color, alpha=0.85, pad=4, lw=0))
    ax_img.set_title("frame 8 — ranked: 1 red, 2 orange, 3 yellow; grey = lower priority; magenta dashed = unlabelled obstacle", fontsize=10)

    # Row 2: map with the same ranks, the side lanes checked and the chosen action
    ax.add_patch(patches.Rectangle((-CORRIDOR_HALF, 0), 2 * CORRIDOR_HALF, CORRIDOR_LEN, color="cyan", alpha=0.25))
    for deg in (-75, -45, -15, 15, 45, 75):
        ax.plot([0, 16 * np.sin(np.radians(deg))], [0, 16 * np.cos(np.radians(deg))], color="lightgrey", lw=0.6)
    for d in (2, 4, 6, 8, 10):
        ax.add_patch(patches.Arc((0, 0), 2 * d, 2 * d, theta1=10, theta2=170, color="lightgrey", lw=0.5))
        ax.text(d * 0.71 + 0.2, d * 0.71, f"{d} m", fontsize=6, color="grey")
    if h.passage:                                               # where the body fits next to the obstacle
        y = h.check_until_m - LANE_MARGIN
        for runs_, col in ((h.free_narrow, "darkorange"), (h.free_wide, "green")):
            for x0, x1 in runs_:
                ax.plot([x0 - BIN / 2, x1 + BIN / 2], [y, y], color=col, lw=7, alpha=0.6, solid_capstyle="butt", zorder=2)
        ax.text(-7.8, 14.3, "bar at the obstacle = where walker fit:\ngreen comfortable, orange narrow", fontsize=7, va="top")
        if pd.notna(h.aim_m):
            ax.plot([0, h.aim_m], [0, y], color="black", lw=1, ls=":")
    ax.scatter(r.lateral_m, r.forward_m, s=14, color="lightgrey", edgecolor="black", lw=0.3, zorder=3, clip_on=True)
    for _, t in top3.iterrows():
        badge(ax, t.lateral_m, t.forward_m, t["rank"], RANK_COLORS[t["rank"]], "white")
    if pd.notna(h.action_clock):
        ang = np.radians(h.action_clock * 30)                     # clock direction -> angle from straight ahead
        ax.annotate("", xy=(2.5 * np.sin(ang), 2.5 * np.cos(ang)), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="black", lw=2.5))
    elif h.level == "stop":
        ax.text(0, 1.2, "STOP", ha="center", fontsize=12, fontweight="bold", color="red")
    ax.plot(0, 0, "k^", ms=10)
    ax.set_xlim(-8, 8); ax.set_ylim(-0.5, 15); ax.set_aspect("equal")
    ax.set_xlabel("left  ←  metres  →  right", fontsize=8); ax.set_ylabel("metres ahead", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title("bird's-eye view — ranked hazards and chosen action", fontsize=10)

    # Key: the hint our module produces vs the WAD reference (display only)
    scores = "\n".join(f"{int(t['rank'])}. {t.prompt:<10} {t.dist_m:4.1f} m  score {t.score:.2f}  {t.motion}"
                       + ("  ◀ decided" if t.trigger else "") for _, t in r.head(5).iterrows())
    hint = "\n".join(textwrap.fill(line, 58, subsequent_indent="   ") for line in h.hint.split("\n"))
    ax_key.text(0, 1, "OUR HINT\n" + hint, transform=ax_key.transAxes, fontsize=8.5, family="monospace", va="top")
    ax_key.text(0, 0.55, "TOP SCORES\n" + scores, transform=ax_key.transAxes, fontsize=7.5, family="monospace", va="top")
    ax_key.text(0, 0.18, "WAD REFERENCE (display only)\n" + "\n".join(textwrap.wrap(REMINDER.get(clip, ""), 48)),
                transform=ax_key.transAxes, fontsize=8, va="top", color="dimgrey")
    fig.suptitle(clip, fontsize=10)
    out = VIZ_DIR / "07_priority" / f"{clip.replace('.frame', '')}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=90, bbox_inches="tight"); plt.show(); plt.close(fig)

for clip in VIZ_CLIPS:
    show_priority(clip)

## 9. Ablations
- Each variant switches off one part of the module; sections 3–8 are not changed.
- The `full` variant must reproduce the saved hints exactly (100%).
- Outputs: `ablations/<variant>_hints.jsonl` and `ablations/<variant>_ranking.parquet`.

In [ ]:
ABLATIONS = {
    "full":         "the frozen module, recomputed here (must match the saved hints exactly)",
    "no_severity":  "every hazard type counts the same (severity = 1)",
    "no_closeness": "distance not used in the score",
    "no_position":  "in the path / beside / to the side not used in the score",
    "no_motion":    "no tracking: motion unknown, no time-to-collision, no 'entering the path' (frame 8 only)",
    "objects_only": "no surface hazards (walls, fences, water, steps) and no unlabelled obstacles",
}
SCORE_PART = {"no_severity": "s_severity", "no_closeness": "s_close", "no_position": "s_path"}
ABL_DIR = OUT_DIR / "ablations"
ABL_DIR.mkdir(parents=True, exist_ok=True)

# candidates exactly as in 8.4 (repeated so 8.4 does not need to be rerun)
obj8 = objects[objects.frame == 8][["clip_id", "track_id", "flat", "ground_x", "ground_y"]]
cand_all = motion8.merge(obj8, on=["clip_id", "track_id"])
keep = (cand_all.conf >= MIN_CONF) | ((cand_all.conf >= MIN_CONF_NEAR) & (cand_all.dist_m <= NEAR_CONF_DIST))
cand_all = cand_all[~cand_all.flat & keep & (cand_all.dist_m <= MAX_DIST) & (cand_all.forward_m > 0)].copy()
cand_all["width_m"] = np.minimum(cand_all.width_m, cand_all.category.map(THIN_WIDTH).fillna(np.inf))
cand_all["in_corridor"] = ((cand_all.forward_m > 0) & (cand_all.forward_m <= CORRIDOR_LEN)
                           & (cand_all.lateral_m.abs() - cand_all.width_m / 2 <= CORRIDOR_HALF))
boxes8 = det[(det.frame == 8) & (det.conf >= MIN_CONF_NEAR)]

def run_variant(name, clip, cands, flags, scene):
    c, fl = cands, dict(flags)
    if name == "objects_only":
        c = c[c.source == "object"].reset_index(drop=True)
    if name == "no_motion":
        c = c.assign(motion="unknown", ttc_s=np.nan, enters_corridor_2s=False)
    scored = score_candidates(c, fl) if len(c) else c
    if name in SCORE_PART and len(scored):
        scored[SCORE_PART[name]] = 1.0 # switch this factor off, then recompute the score
        scored["score"] = scored.s_severity * scored.s_close * scored.s_path * scored.s_motion * scored.s_context * np.sqrt(scored.conf)
    level, move_clock, trig = decide(scored, fl, scene)
    if len(scored):
        scored["trigger"] = scored.index == trig
        ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)
        ranked = ranked.assign(clip_id=clip, rank=np.arange(1, len(ranked) + 1))
    else:
        ranked = scored
    top = ranked.iloc[0] if len(ranked) and level != "clear" else None
    tr = ranked[ranked.trigger].iloc[0] if len(ranked) and ranked.trigger.any() else None
    cause = "" if tr is None else ("parked " if tr.parked else "") + tr.prompt
    row = dict(clip_id=clip, level=level, clear=level == "clear", action_clock=move_clock,
               top_category=None if top is None else top.category, top_clock=None if top is None else int(top.clock),
               trigger_category=None if tr is None else tr.category, passage=fl["passage"],
               on_road=fl["on_road"], crowded=fl["crowded"], mixed_ground=fl["mixed_ground"],
               hint=hint_text(ranked, level, move_clock, fl, cause))
    keep_cols = ["clip_id", "rank", "category", "prompt", "source", "dist_m", "clock", "score", "trigger"]
    return row, (ranked[keep_cols] if len(ranked) else None)

rows = {n: [] for n in ABLATIONS}; ranks = {n: [] for n in ABLATIONS}
t0 = time.time()
for k, clip in enumerate(RUN_CLIPS, 1):
    cands = cand_all[cand_all.clip_id == clip].assign(source="object", fenced=False)
    surf_c, flags, scene = surface_hazards(clip, boxes8[boxes8.clip_id == clip])
    flags["crowded"] = bool(((cands.category == "pedestrian") & (cands.dist_m <= 8)).sum() >= 6)
    if surf_c:
        cands = pd.concat([cands, pd.DataFrame(surf_c)], ignore_index=True)
    cands["fenced"] = cands.fenced.eq(True)
    cands = cands.reset_index(drop=True)
    for name in ABLATIONS:
        row, rk = run_variant(name, clip, cands, flags, scene)
        rows[name].append(row)
        if rk is not None:
            ranks[name].append(rk)
    if k % 100 == 0 or k == len(RUN_CLIPS):
        print(f"  {k}/{len(RUN_CLIPS)} clips | {(time.time() - t0) / 60:.1f} min")

for name in ABLATIONS:
    pd.DataFrame(rows[name]).to_json(ABL_DIR / f"{name}_hints.jsonl", orient="records", lines=True, force_ascii=False)
    pd.concat(ranks[name], ignore_index=True).to_parquet(ABL_DIR / f"{name}_ranking.parquet", index=False)

# check: the 'full' variant must reproduce the saved hints exactly
frozen = pd.read_json(SAVED / "hints.jsonl", lines=True).set_index("clip_id")
full = pd.DataFrame(rows["full"]).set_index("clip_id").reindex(frozen.index)
same = (full.level == frozen.level) & (full.top_category.fillna("-") == frozen.top_category.fillna("-"))
print(f"check: 'full' matches the saved hints in {same.mean():.1%} of clips (must be 100%)")

# decision levels per variant
summary = pd.DataFrame({n: pd.DataFrame(rows[n]).level.value_counts() for n in ABLATIONS}).fillna(0).astype(int)
summary.reindex(["stop", "move", "slow", "caution", "clear"])

## Save and download
- `<subset>_outputs.zip`: hints, ranking, baselines and ablations (what the evaluation notebook needs).
- `<subset>_results.zip`: outputs + perception cache (to resume or redraw figures later).

In [ ]:
W = Path("/kaggle/working")

def zip_dirs(name, roots):
    with zipfile.ZipFile(W / name, "w", zipfile.ZIP_DEFLATED) as z:
        for root in roots:
            for p in (W / root).rglob("*"):
                if p.is_file() and "viz" not in p.parts:
                    z.write(p, p.relative_to(W))

zip_dirs(f"{SUBSET}_outputs.zip", [f"outputs/{SUBSET}"])
zip_dirs(f"{SUBSET}_results.zip", [f"outputs/{SUBSET}", f"cache/{SUBSET}"])
display(FileLink(f"{SUBSET}_outputs.zip"), FileLink(f"{SUBSET}_results.zip"))